# Spectral-mask sweep: balanced joint sampling + local refinement

300 complete masks, then up to 150 local proposals for each of **LP01 D_NL** and **LP01 D_IO**. The 100 field-amplitude factors use `{0, 0.2, 0.4, 0.6, 0.8, 1}`. The original LLMBO and gradient-descent notebooks are preserved.

**Result status comes from the validation section below.** Search-grid results remain preliminary when a required check fails.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebook_helpers" / "run_library.py").is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebook_helpers"))

import json
import numpy as np
import pandas as pd
from IPython.display import display, Image
from llm_bo.spectral_mask_balanced import run_experiment, load_run
from llm_bo.spectral_mask_balanced_report import build_report
from notebook_helpers.run_library import file_saved_run

SAVED_RUN = None  # Use an existing local campaign directory to reopen it.
if SAVED_RUN is None:
    run = run_experiment(seed=42, max_minutes=90)
else:
    run = load_run(SAVED_RUN)
SAVED_RUN = run.directory
file_saved_run(SAVED_RUN, "Spectral amplitude mask", "Balanced sampling and local search")
print("Run directory:", SAVED_RUN)
print("Stop reason:", run.result["stop_reason"])
report_ready = all(run.result.get("validation", {}).get(metric, {}).get("selected_refined")
                   for metric in ("D_NL", "D_IO"))
if report_ready:
    build_report(SAVED_RUN)
else:
    print("The time budget ended before the full report was available. Measurements are in state.json.")

## 1. Keep the physics and define both objectives

Start with the same **100 kW, 200 fs Gaussian**, three equal-amplitude modes and zero relative phase. Apply one shared amplitude mask over 100 equal-width angular-frequency bands spanning the original 99.9%-energy bandwidth. Keep the spectral tails unchanged; do not renormalize the shaped pulse.

`D_NL` compares the matched linear and nonlinear outputs of LP01. `D_IO` compares the candidate input and nonlinear output of LP01. Both use normalized complex overlap, with no fitted time shift. The hard brightness floor is **50% of the unmasked all-mode nonlinear output photon count**. The two scores come from the same saved input/linear/nonlinear fields.

In [ ]:
if report_ready:
    spec = json.loads((SAVED_RUN / "run.json").read_text())
    problem_card = {
        "pulse": "100 kW before masking; 200 fs; three equal-amplitude, zero-phase modes",
        "fiber": "2.86 m; 52.5 µm core radius; NA 0.1; carrier 1379.2 nm",
        "mask": "100 shared amplitude factors; six levels; unchanged tails; no renormalization",
        "objectives": "minimize LP01 D_NL and D_IO separately",
        "constraint": "all-mode output photons / unmasked reference >= 0.50",
        "search_grid": "Nt = 16384; 5 mm propagation step",
        "budget": "300 balanced + 150 local per objective; 7 extra baseline/check pairs; 90 min",
    }
    display(pd.Series(problem_card, name="Setup").to_frame())

## 2. Construct the balanced batch

Each column is one factor and each row is one complete mask. Every factor receives **50 copies of every allowed value**, shuffled independently with seed 42. This gives exact individual-factor balance; it does not cover all interactions. The 300 sampled masks remain unchanged even when they violate brightness.

These factors multiply **field amplitude**, so energy scales with the factor squared. The mean squared factor is 0.3667: this balanced batch transmits about 36.7% of the original input energy on average, including the unchanged tails. That input-energy calculation does not replace the measured output-photon check.

In [ ]:
if report_ready:
    rng = np.random.default_rng(spec["seed"])
    level_ids = np.column_stack([
        rng.permutation(np.repeat(np.arange(6), 50))
        for _ in range(100)
    ])
    masks = level_ids / 5

    counts = np.array([(level_ids == k).sum(axis=0) for k in range(6)])
    assert masks.shape == (300, 100)
    assert np.all(counts == 50)
    assert np.array_equal(masks, np.load(SAVED_RUN / "balanced_masks.npy"))
    print("300 masks × 100 factors; exactly 50 uses of each level per factor.")
    joint = run.result["joint"]
    print(sum(r["feasible"] for r in joint), "/", len(joint), "measured masks meet the brightness floor.")
    display(pd.read_csv(SAVED_RUN / "coverage_summary.csv"))

## 3. Evaluate, keep feasible improvements, repeat

The controller evaluates every balanced mask once and scores both objectives. It chooses up to three diverse feasible starts for each objective. When there are too few, it restores sampled masks by raising each factor by 0.2 toward unity until the measured brightness passes. **Restoration trials are adaptive trials and count inside that objective’s 150-proposal allowance.**

Local proposals change **2–10 bands by ±0.2**. Each chain keeps a proposal only if it is feasible and improves that chain’s objective. Repeated evaluations use the cache. All attempts, acceptances, rejections, masks and elapsed times are saved.

The short controller loop is:

```python
# sample runs one matched pair and records both LP01 scores.
for index, mask in enumerate(masks):
    campaign.sample(index, mask)

for objective in ("D_NL", "D_IO"):
    # start_local selects feasible starts and restores them when needed.
    campaign.start_local(objective)
    for step in range(remaining_local_trials):
        # local_step tests one proposal and keeps only a feasible improvement.
        proposal, accepted = campaign.local_step(objective, step, rng)
```

The cell below replays the acceptance check against the saved measurements; it does not rerun the search.

In [ ]:
if report_ready:
    # accepts checks feasibility and a strict improvement in the chosen objective.
    from llm_bo.spectral_mask_balanced import accepts

    observations = {r["evaluation"]: r for r in run.result["observations"]}
    for objective in ("D_NL", "D_IO"):
        local = run.result["local"][objective]
        accepted_count = 0
        for decision in local["proposals"]:
            current = observations[decision["from_evaluation"]]
            candidate = observations[decision["evaluation"]]
            accepted = accepts(current, candidate, objective)
            assert accepted == decision["accepted"]
            if accepted:
                accepted_count += 1
        print(objective, ":", len(local["restoration"]), "restoration trials,",
              len(local["proposals"]), "local proposals,", accepted_count, "accepted")
    print("Stop:", run.result["stop_reason"])

### Optimization paths

Each loop has one path figure. Proposal order gives direction; the local route shows accepted chains separately from rejected proposals. Diamonds mark each chain’s start and last accepted state. The mask heatmap shows the best mask across chains: a change of winning chain can change many bands at once, even though each tested local move changes only 2–10 bands. Fine-check points are separate from search-grid paths.

In [ ]:
if report_ready:
    for name in ("joint_sampling_path", "d_nl_local_path", "d_io_local_path", "convergence_elapsed"):
        display(Image(filename=str(SAVED_RUN / f"{name}.png")))

## 4. Recheck both selected masks

Use the original finer-check procedure: **5 mm → 1 mm** propagation steps on Nt = 16384, then **Nt = 65536 with twice the time window** at 1 mm. The allowed absolute objective difference is **2×10⁻⁴**. The refined brightness denominator is the unmasked nonlinear baseline on the matching refined grid.

A failed or missing check blocks promotion to a numerically validated result. Each branch’s acceptance checks apply to its optimized metric; the other reported score is a diagnostic. No claim of a global optimum or local convergence follows from using up the proposal budget.

In [ ]:
if report_ready:
    display(pd.read_csv(SAVED_RUN / "selected_results.csv"))
    display(pd.read_csv(SAVED_RUN / "validation_checks.csv"))
    display(pd.read_csv(SAVED_RUN / "reported_metric_grid_changes.csv"))

## 5. Compare input, linear output and nonlinear output

Columns show the unmasked baseline and the two selected masks, all on the finer time grid. Time traces are in the first row and wavelength spectra in the second. Each case uses its own matched input/linear/nonlinear fields, with shared scales. Spectral energy density is in **nJ/nm**, including the frequency-to-wavelength Jacobian. No per-curve normalization or time alignment is applied. Scores use the full grids.

Where the input and linear spectra coincide, the linear-output curve covers the input curve.

**LP01**, the optimized mode:

In [ ]:
if report_ready:
    display(Image(filename=str(SAVED_RUN / "pulse_comparison_lp01.png")))

All propagated modes are retained. The all-mode plot is spatially integrated power; it is not a scalar sum of modal amplitudes.

In [ ]:
if report_ready:
    for name in ("lp11_cos", "lp11_sin", "all_modes"):
        display(Image(filename=str(SAVED_RUN / f"pulse_comparison_{name}.png")))

Full-window LP01 traces expose any low-power tails hidden by the central ±2 ps view:

In [ ]:
if report_ready:
    display(Image(filename=str(SAVED_RUN / "pulse_comparison_lp01_full_window.png")))

## 6. Measured runtime and compute cost

The table reads the saved measurements from `run_costs.json`. One objective pair contains **one nonlinear GNLSE solve plus one matched linear solve**; cache hits add no solves. The campaign timer includes baseline setup, search and finer checks. Timed verification/reporting is listed separately from elapsed time including review and waiting.

CPU time and sampled memory cover the controller and Julia process. **The solver uses the CPU.** Device-wide GPU counters include other applications and cannot be attributed to this run. Memory is a sampled peak; storage is the recorded inventory, with GB = 10⁹ bytes. Electricity and monetary costs were not measured. Implementation before campaign creation was not timed. These are the original run/export costs; later notebook edits do not restart those timers.

In [ ]:
if report_ready:
    cost = json.loads((SAVED_RUN / "run_costs.json").read_text())
    display(pd.Series(cost, name="Measured cost").to_frame())
    display(Image(filename=str(SAVED_RUN / "measured_resources.png")))